# 05 — Statistical validation (fast, publication-oriented)

This notebook performs the statistical validation **from the completed `02_quality.csv` table**.
It does **not** re-run PM4Py models and therefore avoids the multi-hour trace-level bootstrap.

It provides:

- Friedman omnibus tests across Alpha, Heuristic, and Inductive Miner;
- Wilcoxon signed-rank pairwise comparisons;
- Holm correction for multiple pairwise tests;
- Kendall's W effect size;
- rank-biserial effect sizes;
- faculty-cluster bootstrap 95% confidence intervals for algorithm means and paired differences.

For final manuscript use, first complete Notebook 02 for **all seven faculties**.

In [1]:

!pip install -q scipy statsmodels 2>/dev/null

import os, pathlib, sys, json
import pandas as pd
import numpy as np

from scipy.stats import friedmanchisquare, wilcoxon, rankdata
from statsmodels.stats.multitest import multipletests

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    RESULTS_DIR = "/content/drive/MyDrive/ProcessMining/results"
except Exception:
    RESULTS_DIR = "/content/ProcessMining/results"

pathlib.Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

QUALITY_FILE = os.path.join(RESULTS_DIR, "02_quality.csv")
print("Python:", sys.version.split()[0])
print("Results folder:", RESULTS_DIR)
print("Quality file:", QUALITY_FILE)

Mounted at /content/drive
Python: 3.13.15
Results folder: /content/drive/MyDrive/ProcessMining/results
Quality file: /content/drive/MyDrive/ProcessMining/results/02_quality.csv


In [2]:


if not os.path.exists(QUALITY_FILE):
    raise FileNotFoundError(
        f"{QUALITY_FILE} not found. Complete 02_model_quality_colab_safe.ipynb "
        "for each faculty first so that it builds the master 02_quality.csv."
    )

quality = pd.read_csv(QUALITY_FILE)

if "error" in quality.columns:
    q = quality[quality["error"].isna()].copy()
else:
    q = quality.copy()


q["faculty"] = q["faculty"].astype(str).str.strip().str.upper()
q["role"] = q["role"].astype(str).str.strip().str.title()
q["notion"] = q["notion"].astype(str).str.strip().str.lower()
q["miner"] = q["miner"].astype(str).str.strip().str.lower()

EXPECTED_FACULTIES = ["FEB", "FIF", "FIK", "FIT", "FKB", "FRI", "FTE"]
EXPECTED_ROLES = ["Student", "Lecturer"]
EXPECTED_MINERS = ["alpha", "heuristic", "inductive"]

print(f"Rows in master table: {len(q)}")
print("Faculties:", sorted(q["faculty"].dropna().unique()))
print("Roles:", sorted(q["role"].dropna().unique()))
print("Case notions:", sorted(q["notion"].dropna().unique()))
print("Miners:", sorted(q["miner"].dropna().unique()))

missing_fac = [f for f in EXPECTED_FACULTIES if f not in set(q["faculty"])]
missing_roles = [r for r in EXPECTED_ROLES if r not in set(q["role"])]

if missing_fac:
    print("\nWARNING — missing faculties:", ", ".join(missing_fac))
else:
    print("\nAll seven faculties are represented.")

if missing_roles:
    print("WARNING — missing role partition(s):", ", ".join(missing_roles))
    print("Statistics will still run for the role(s) present, but this is not full cross-role validation.")

audit = (
    q.groupby(["notion", "miner"])
     .agg(rows=("faculty", "size"),
          faculties=("faculty", "nunique"),
          roles=("role", "nunique"),
          fitness_nonmissing=("fitness", "count"),
          precision_nonmissing=("precision", "count"),
          generalization_nonmissing=("generalization", "count"),
          simplicity_nonmissing=("simplicity", "count"))
     .reset_index()
)
audit

Rows in master table: 42
Faculties: ['FEB', 'FIF', 'FIK', 'FIT', 'FKB', 'FRI', 'FTE']
Roles: ['Student']
Case notions: ['course', 'user']
Miners: ['alpha', 'heuristic', 'inductive']

All seven faculties are represented.
WARNING — missing role partition(s): Lecturer
Statistics will still run for the role(s) present, but this is not full cross-role validation.


,notion,miner,rows,faculties,roles,fitness_nonmissing,precision_nonmissing,generalization_nonmissing,simplicity_nonmissing
0,course,alpha,7,7,1,7,7,7,7
1,course,heuristic,7,7,1,7,7,7,7
2,course,inductive,7,7,1,7,7,7,7
3,user,alpha,7,7,1,7,0,7,7
4,user,heuristic,7,7,1,7,0,7,7
5,user,inductive,7,7,1,7,0,7,7


## Paired omnibus and pairwise tests

The observational unit is a **faculty-role partition**. All three miners must have a
non-missing value for a partition to enter a paired comparison.

Because ETC precision is deliberately skipped for long user-level traces in Notebook 02,
precision testing is expected to be available primarily for the **course-level case notion**.

In [3]:


stat_rows = []

for notion in sorted(q["notion"].dropna().unique()):
    sub = q[q["notion"] == notion]
    print(f"\n{'='*72}\n{notion.upper()}-LEVEL CASE NOTION\n{'='*72}")

    for dim in ["fitness", "precision", "generalization", "simplicity"]:



        piv = (
            sub.pivot_table(index=["faculty", "role"], columns="miner",
                            values=dim, aggfunc="first")
               .reindex(columns=EXPECTED_MINERS)
        )


        piv = piv.dropna(how="any", subset=EXPECTED_MINERS)

        if len(piv) < 3:
            print(
                f"\n{dim}: only {len(piv)} complete paired partition(s); skipped. "
                "This is expected for user-level precision when it was not computed."
            )
            continue

        a, h, i = piv["alpha"], piv["heuristic"], piv["inductive"]

        stat, p = friedmanchisquare(a, h, i)
        k, n = 3, len(piv)
        W = stat / (n * (k - 1))

        stat_rows.append(dict(
            notion=notion, dimension=dim, test="Friedman",
            comparison="alpha vs heuristic vs inductive",
            n=n, statistic=float(stat), p_raw=float(p), p_holm=np.nan,
            effect_name="Kendall_W", effect=float(W)
        ))

        print(f"\n{dim} (n={n} faculty-role partitions)")
        print(f"  Friedman chi2={stat:.4f}, p={p:.4g}, Kendall W={W:.4f}")
        print("  means: " + ", ".join(
            f"{m}={piv[m].mean():.4f}" for m in EXPECTED_MINERS
        ))

        pairs = [("alpha", "heuristic"),
                 ("alpha", "inductive"),
                 ("heuristic", "inductive")]

        pair_results = []
        for x, y in pairs:
            d = piv[x] - piv[y]
            try:
                ws, wp = wilcoxon(piv[x], piv[y], zero_method="wilcox")
            except ValueError:
                ws, wp = np.nan, np.nan

            nz = d[d != 0]
            if len(nz):
                ranks = rankdata(np.abs(nz))
                rbc = float((ranks * np.sign(nz)).sum() /
                            (len(nz) * (len(nz) + 1) / 2))
            else:
                rbc = 0.0

            pair_results.append([x, y, ws, wp, rbc])

        raw_p = np.array([r[3] for r in pair_results], dtype=float)
        ok = np.isfinite(raw_p)
        holm = np.full(len(raw_p), np.nan)
        if ok.any():
            holm[ok] = multipletests(raw_p[ok], method="holm")[1]

        for (x, y, ws, wp, rbc), hp in zip(pair_results, holm):
            stat_rows.append(dict(
                notion=notion, dimension=dim,
                test="Wilcoxon signed-rank",
                comparison=f"{x} vs {y}",
                n=n, statistic=float(ws) if np.isfinite(ws) else np.nan,
                p_raw=float(wp) if np.isfinite(wp) else np.nan,
                p_holm=float(hp) if np.isfinite(hp) else np.nan,
                effect_name="rank_biserial", effect=rbc
            ))
            print(f"    {x:>9} vs {y:<9} "
                  f"p={wp:.4g}  Holm={hp:.4g}  RBC={rbc:+.3f}")

stats = pd.DataFrame(stat_rows)
STATS_OUT = os.path.join(RESULTS_DIR, "05_algorithm_stats.csv")
stats.to_csv(STATS_OUT, index=False)
print("\nSaved:", STATS_OUT)
stats


COURSE-LEVEL CASE NOTION

fitness (n=7 faculty-role partitions)
  Friedman chi2=14.0000, p=0.0009119, Kendall W=1.0000
  means: alpha=0.1047, heuristic=0.9770, inductive=0.9955
        alpha vs heuristic p=0.01562  Holm=0.04688  RBC=-1.000
        alpha vs inductive p=0.01562  Holm=0.04688  RBC=-1.000
    heuristic vs inductive p=0.01562  Holm=0.04688  RBC=-1.000

precision (n=7 faculty-role partitions)
  Friedman chi2=10.5714, p=0.005063, Kendall W=0.7551
  means: alpha=0.1470, heuristic=0.1408, inductive=0.0548
        alpha vs heuristic p=0.9375  Holm=0.9375  RBC=+0.071
        alpha vs inductive p=0.01562  Holm=0.04688  RBC=+1.000
    heuristic vs inductive p=0.01562  Holm=0.04688  RBC=+1.000

generalization (n=7 faculty-role partitions)
  Friedman chi2=14.0000, p=0.0009119, Kendall W=1.0000
  means: alpha=0.7300, heuristic=0.6664, inductive=0.7936
        alpha vs heuristic p=0.01562  Holm=0.04688  RBC=+1.000
        alpha vs inductive p=0.01562  Holm=0.04688  RBC=-1.000
    heur

,notion,dimension,test,comparison,n,statistic,p_raw,p_holm,effect_name,effect
0,course,fitness,Friedman,alpha vs heuristic vs inductive,7,14.000000,0.000912,NaN,Kendall_W,1.000000
1,course,fitness,Wilcoxon signed-rank,alpha vs heuristic,7,0.000000,0.015625,0.046875,rank_biserial,-1.000000
2,course,fitness,Wilcoxon signed-rank,alpha vs inductive,7,0.000000,0.015625,0.046875,rank_biserial,-1.000000
3,course,fitness,Wilcoxon signed-rank,heuristic vs inductive,7,0.000000,0.015625,0.046875,rank_biserial,-1.000000
4,course,precision,Friedman,alpha vs heuristic vs inductive,7,10.571429,0.005063,NaN,Kendall_W,0.755102
5,course,precision,Wilcoxon signed-rank,alpha vs heuristic,7,13.000000,0.937500,0.937500,rank_biserial,0.071429
6,course,precision,Wilcoxon signed-rank,alpha vs inductive,7,0.000000,0.015625,0.046875,rank_biserial,1.000000
7,course,precision,Wilcoxon signed-rank,heuristic vs inductive,7,0.000000,0.015625,0.046875,rank_biserial,1.000000
8,course,generalization,Friedman,alpha vs heuristic vs inductive,7,14.000000,0.000912,NaN,Kendall_W,1.000000
9,course,generalization,Wilcoxon signed-rank,alpha vs heuristic,7,0.000000,0.015625,0.046875,rank_biserial,1.000000


## Fast faculty-cluster bootstrap confidence intervals

This bootstrap resamples **faculties**, not individual events or traces. When a faculty is
drawn, both lecturer and student partitions for that faculty are retained. This preserves
the within-faculty dependence between roles while quantifying uncertainty in the
cross-faculty mean.

This is much faster than rediscovering and replaying process models hundreds of times.

In [4]:


N_BOOT = 10000
BOOT_SEED = 20260828

rng = np.random.default_rng(BOOT_SEED)
ci_rows = []
diff_rows = []

pairs = [("alpha", "heuristic"),
         ("alpha", "inductive"),
         ("heuristic", "inductive")]

for notion in sorted(q["notion"].dropna().unique()):
    sub_n = q[q["notion"] == notion].copy()

    for dim in ["fitness", "precision", "generalization", "simplicity"]:
        wide = (
            sub_n.pivot_table(index=["faculty", "role"], columns="miner",
                              values=dim, aggfunc="first")
                 .reindex(columns=EXPECTED_MINERS)
        )



        if wide.dropna(how="all").empty:
            print(f"Skipping {notion}/{dim}: no computed values.")
            continue

        piv = wide.reset_index()
        facs = sorted(piv["faculty"].dropna().unique())

        if len(facs) < 3:
            print(f"Skipping {notion}/{dim}: only {len(facs)} faculties.")
            continue

        boot_means = {m: [] for m in EXPECTED_MINERS}
        boot_diffs = {p: [] for p in pairs}

        for _ in range(N_BOOT):
            sampled_facs = rng.choice(facs, size=len(facs), replace=True)
            pieces = []
            for draw_id, fac in enumerate(sampled_facs):
                z = piv[piv["faculty"] == fac].copy()
                z["_draw"] = draw_id
                pieces.append(z)
            bdf = pd.concat(pieces, ignore_index=True)

            for m in EXPECTED_MINERS:
                vals = bdf[m].dropna().to_numpy(float)
                if len(vals):
                    boot_means[m].append(vals.mean())

            for x, y in pairs:
                pair_df = bdf[[x, y]].dropna()
                if len(pair_df):
                    boot_diffs[(x, y)].append((pair_df[x] - pair_df[y]).mean())

        for m, vals in boot_means.items():
            if len(vals) >= 100:
                v = np.asarray(vals)
                point = piv[m].dropna().mean()
                ci_rows.append(dict(
                    notion=notion, dimension=dim, miner=m,
                    faculties=len(facs),
                    observed_partitions=int(piv[m].notna().sum()),
                    n_boot=N_BOOT, seed=BOOT_SEED,
                    mean=float(point),
                    ci_lo=float(np.percentile(v, 2.5)),
                    ci_hi=float(np.percentile(v, 97.5))
                ))

        for (x, y), vals in boot_diffs.items():
            if len(vals) >= 100:
                v = np.asarray(vals)
                pair_df = piv[[x, y]].dropna()
                point = (pair_df[x] - pair_df[y]).mean()
                diff_rows.append(dict(
                    notion=notion, dimension=dim,
                    comparison=f"{x} - {y}",
                    faculties=len(facs),
                    observed_paired_partitions=len(pair_df),
                    n_boot=N_BOOT, seed=BOOT_SEED,
                    mean_difference=float(point),
                    ci_lo=float(np.percentile(v, 2.5)),
                    ci_hi=float(np.percentile(v, 97.5))
                ))

ci = pd.DataFrame(ci_rows)
diff_ci = pd.DataFrame(diff_rows)

CI_OUT = os.path.join(RESULTS_DIR, "05_algorithm_mean_bootstrap_ci.csv")
DIFF_OUT = os.path.join(RESULTS_DIR, "05_pairwise_difference_bootstrap_ci.csv")
ci.to_csv(CI_OUT, index=False)
diff_ci.to_csv(DIFF_OUT, index=False)

print("Saved:", CI_OUT)
print("Saved:", DIFF_OUT)
print("\nAlgorithm mean CIs:")
display(ci.round(4))
print("\nPairwise mean-difference CIs:")
display(diff_ci.round(4))

Skipping user/precision: no computed values.
Saved: /content/drive/MyDrive/ProcessMining/results/05_algorithm_mean_bootstrap_ci.csv
Saved: /content/drive/MyDrive/ProcessMining/results/05_pairwise_difference_bootstrap_ci.csv

Algorithm mean CIs:


,notion,dimension,miner,faculties,observed_partitions,n_boot,seed,mean,ci_lo,ci_hi
0,course,fitness,alpha,7,7,10000,20260828,0.1047,0.0932,0.1163
1,course,fitness,heuristic,7,7,10000,20260828,0.9770,0.9719,0.9821
2,course,fitness,inductive,7,7,10000,20260828,0.9955,0.9903,0.9996
3,course,precision,alpha,7,7,10000,20260828,0.1470,0.1226,0.1732
4,course,precision,heuristic,7,7,10000,20260828,0.1408,0.1225,0.1616
5,course,precision,inductive,7,7,10000,20260828,0.0548,0.0501,0.0594
6,course,generalization,alpha,7,7,10000,20260828,0.7300,0.7095,0.7479
7,course,generalization,heuristic,7,7,10000,20260828,0.6664,0.6506,0.6849
8,course,generalization,inductive,7,7,10000,20260828,0.7936,0.7793,0.8111
9,course,simplicity,alpha,7,7,10000,20260828,1.0000,1.0000,1.0000



Pairwise mean-difference CIs:


,notion,dimension,comparison,faculties,observed_paired_partitions,n_boot,seed,mean_difference,ci_lo,ci_hi
0,course,fitness,alpha - heuristic,7,7,10000,20260828,-0.8723,-0.8842,-0.8610
1,course,fitness,alpha - inductive,7,7,10000,20260828,-0.8909,-0.9009,-0.8806
2,course,fitness,heuristic - inductive,7,7,10000,20260828,-0.0185,-0.0261,-0.0114
3,course,precision,alpha - heuristic,7,7,10000,20260828,0.0062,-0.0294,0.0441
4,course,precision,alpha - inductive,7,7,10000,20260828,0.0922,0.0713,0.1152
5,course,precision,heuristic - inductive,7,7,10000,20260828,0.0860,0.0658,0.1072
6,course,generalization,alpha - heuristic,7,7,10000,20260828,0.0636,0.0433,0.0857
7,course,generalization,alpha - inductive,7,7,10000,20260828,-0.0635,-0.0839,-0.0430
8,course,generalization,heuristic - inductive,7,7,10000,20260828,-0.1272,-0.1425,-0.1116
9,course,simplicity,alpha - heuristic,7,7,10000,20260828,0.5300,0.5274,0.5327


In [5]:


complete_faculties = sorted(set(q["faculty"]) & set(EXPECTED_FACULTIES))
roles_present = sorted(set(q["role"]) & set(EXPECTED_ROLES))

print("Faculties represented:", len(complete_faculties), "/ 7",
      ", ".join(complete_faculties))
print("Roles represented:", ", ".join(roles_present) if roles_present else "none")

course_precision = (
    q[q["notion"].eq("course")]
      .pivot_table(index=["faculty","role"], columns="miner",
                   values="precision", aggfunc="first")
      .reindex(columns=EXPECTED_MINERS)
)
n_course_prec = len(course_precision.dropna(how="any", subset=EXPECTED_MINERS))
expected_for_present_roles = 7 * len(roles_present)

print("Complete course-level precision faculty-role partitions:",
      n_course_prec, "/", expected_for_present_roles)

full_role_coverage = set(EXPECTED_ROLES).issubset(set(roles_present))

if (
    len(complete_faculties) == 7
    and full_role_coverage
    and n_course_prec == 14
):
    print("\nREADY: full seven-faculty, two-role statistical validation is complete.")
elif len(complete_faculties) == 7 and roles_present == ["Student"] and n_course_prec == 7:
    print("\nSTUDENT-ONLY ANALYSIS COMPLETE:")
    print("All seven faculties are represented for Student, including course-level precision.")
    print("Lecturer partitions are absent, so do not describe these tests as full cross-role validation.")
else:
    print("\nNOT YET FINAL:")
    print("Check the audit above for missing faculties, roles, or course-level precision.")

print("\nFiles written to:", RESULTS_DIR)
for name in [
    "05_algorithm_stats.csv",
    "05_algorithm_mean_bootstrap_ci.csv",
    "05_pairwise_difference_bootstrap_ci.csv"
]:
    print(" -", name)

Faculties represented: 7 / 7 FEB, FIF, FIK, FIT, FKB, FRI, FTE
Roles represented: Student
Complete course-level precision faculty-role partitions: 7 / 7

STUDENT-ONLY ANALYSIS COMPLETE:
All seven faculties are represented for Student, including course-level precision.
Lecturer partitions are absent, so do not describe these tests as full cross-role validation.

Files written to: /content/drive/MyDrive/ProcessMining/results
 - 05_algorithm_stats.csv
 - 05_algorithm_mean_bootstrap_ci.csv
 - 05_pairwise_difference_bootstrap_ci.csv
